# LSTM (Long Short-Term Memory) & GRU (Gated Recurrent Unit) (gates)

_Deep Learning_

**Add little gates that decide what to remember and what to forget over long sequences.**

Plain RNNs (Recurrent Neural Networks) forget the distant past because of vanishing gradients.

     LSTM and GRU are smarter RNN cells. They add gates: little controls that decide what to keep, what to forget, and what to output.

---

This notebook is a practice scaffold for the **LSTM (Long Short-Term Memory) & GRU (Gated Recurrent Unit) (gates)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Create an LSTM and a GRU cell

PyTorch ships both gated recurrent cells ready-made. Each takes 3-dimensional inputs and keeps an 8-dimensional hidden state. `batch_first=True` means we feed tensors shaped `(batch, timesteps, features)`. We will run the same input through both to compare what they return.

In [ ]:
import torch
import torch.nn as nn

lstm = nn.LSTM(input_size=3, hidden_size=8, batch_first=True)
gru = nn.GRU(input_size=3, hidden_size=8, batch_first=True)

### Step 2 — Run a batch of sequences through each cell

We push the same input — 2 sequences, 10 timesteps each, 3 features per step — through both. The key difference shows up in the return values: the **LSTM** returns two states, a hidden state `h` and a separate **cell state `c`** (its protected long-term memory), while the **GRU** returns just a single hidden state.

In [ ]:
x = torch.randn(2, 10, 3)        # 2 sequences, 10 steps, 3 features

out_l, (h_l, c_l) = lstm(x)      # LSTM carries hidden state h AND cell state c
out_g, h_g = gru(x)              # GRU has just one hidden state

### Step 3 — Inspect the shapes and parameter count

The output sequences from both have shape `(2, 10, 8)` — one 8-dim hidden vector per timestep. The LSTM additionally exposes the cell state `c`, the long-term memory its forget gate protects. Counting parameters shows the LSTM is heavier: it has four gates' worth of weights, where the GRU uses three.

In [ ]:
print("LSTM out:", tuple(out_l.shape), " cell:", tuple(c_l.shape))
print("GRU  out:", tuple(out_g.shape))

# the cell state c is the long-term memory the forget gate protects
lstm_params = sum(p.numel() for p in lstm.parameters())
print("LSTM params:", lstm_params)

## Visualize the data & results

_Why do LSTM/GRU gates beat a plain RNN at carrying information over a long sequence?_

### Step 1 — Model how much signal each cell retains per step

We illustrate the vanishing-gradient problem with a simple multiplicative model. Over 20 timesteps, a plain RNN effectively multiplies the carried signal by about 0.7 each step, while a gated cell holds it almost unchanged (factor ~0.983). Raising these factors to the power `t` gives the fraction of the original signal still present at step `t`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.arange(0, 20)

rnn = 0.7 ** t                            # plain RNN multiplies by ~0.7 each step
gated = 0.983 ** t                        # gated cell holds info nearly unchanged

### Step 2 — Plot retained signal across the sequence

The plain-RNN curve decays toward zero within a handful of steps — that is the distant past being forgotten. The gated LSTM/GRU curve stays high across all 20 steps, which is exactly why gates let these cells carry information over long sequences.

In [ ]:
plt.plot(t, rnn, color="#ff7b72", label="plain RNN")
plt.plot(t, gated, color="#7ee787", label="LSTM/GRU (gated)")
plt.title("Information retained across a long sequence")
plt.xlabel("timestep")
plt.ylabel("retained signal")
plt.legend()
plt.show()